In [1]:
from collections import defaultdict, Counter
from typing import Dict, List, Tuple

In [2]:
class BPETokenizer:
    def __init__(self, vocab_size: int, min_frequency: int = 2):
        self.vocab_size = vocab_size
        self.min_frequency = min_frequency
        self.merges: Dict[Tuple[str, str], str] = {}
        self.vocab: List[str] = []
        self.word_cache: Dict[str, List[str]] = {}

    def compute_pair_freqs(self, word_freqs: Dict[str, int]) -> Dict[Tuple[str, str], int]:
        """Вычисление частот пар символов/токенов"""
        pair_freqs = defaultdict(int)
        for word, freq in word_freqs.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pair = (symbols[i], symbols[i + 1])
                pair_freqs[pair] += freq
        return dict(pair_freqs)

    def merge_pair(self, pair: Tuple[str, str], word_freqs: Dict[str, int]) -> Dict[str, int]:
        """Слияние пары токенов"""
        new_word_freqs = {}
        merged_token = ''.join(pair)

        for word, freq in word_freqs.items():
            parts = word.split()
            new_parts = []
            i = 0
            while i < len(parts):
                if i < len(parts) - 1 and (parts[i], parts[i + 1]) == pair:
                    new_parts.append(merged_token)
                    i += 2
                else:
                    new_parts.append(parts[i])
                    i += 1
            new_word = " ".join(new_parts)
            new_word_freqs[new_word] = freq

        return new_word_freqs

    def train(self, text: str) -> None:
        """Обучение токенизатора на тексте"""
        words = text.split()
        word_freqs = Counter(words)

        # Разбиваем слова на символы и инициализируем словарь
        words = {" ".join(list(word)): freq for word, freq in word_freqs.items()}
        vocab = set(char for word in words for char in word.split())

        while len(vocab) < self.vocab_size:
            pair_freqs = self.compute_pair_freqs(words)
            if not pair_freqs:
                break

            best_pair = max(pair_freqs, key=pair_freqs.get)
            if pair_freqs[best_pair] < self.min_frequency:
                break

            words = self.merge_pair(best_pair, words)
            self.merges[best_pair] = ''.join(best_pair)
            vocab.add(''.join(best_pair))

        self.vocab = list(vocab)
        print(f"Обучение завершено. Создано {len(self.vocab)} токенов.")

    def tokenize(self, word: str) -> List[str]:
        """Токенизация слова с учетом обученных слияний"""
        if word in self.word_cache:
            return self.word_cache[word]

        tokens = list(word)
        word_str = " ".join(tokens)

        for pair, merged in self.merges.items():
            bigram = " ".join(pair)
            replacement = merged
            parts = word_str.split()
            i = 0
            new_parts = []

            while i < len(parts):
                if i < len(parts) - 1 and (parts[i], parts[i + 1]) == pair:
                    new_parts.append(replacement)
                    i += 2
                else:
                    new_parts.append(parts[i])
                    i += 1

            word_str = " ".join(new_parts)

        result = word_str.split()
        self.word_cache[word] = result
        return result


In [3]:
def read_text_from_file(file_path: str) -> str:
    """Чтение текста из файла"""
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

def preprocess_text(text: str) -> str:
    """Предварительная обработка текста"""
    return text.lower().replace('\n', ' ').strip()

def save_tokens_to_file(tokens: List[str], output_path: str) -> None:
    """Сохранение токенов в файл"""
    with open(output_path, 'w', encoding='utf-8') as file:
        file.write(' '.join(tokens))

In [4]:
def bpe_pipeline(input_file: str, output_file: str, vocab_size: int = 100, min_frequency: int = 2):
    """Pipeline для токенизации текста с использованием BPE"""
    text = read_text_from_file(input_file)
    text = preprocess_text(text)

    tokenizer = BPETokenizer(vocab_size, min_frequency)
    tokenizer.train(text)

    # Токенизируем каждое слово отдельно
    tokens = []
    for word in text.split():
        tokens.extend(tokenizer.tokenize(word))

    save_tokens_to_file(tokens, output_file)
    print(f"Токенизация завершена. Результат сохранен в {output_file}")

In [5]:
# Пример использования
input_file = '/content/Токенизация_1_processed.txt'
output_file = '/content/tokenized_output.txt'
bpe_pipeline(input_file, output_file, vocab_size=150)

Обучение завершено. Создано 150 токенов.
Токенизация завершена. Результат сохранен в /content/tokenized_output.txt
